# Chronicle — Merkle-DAG Agent Traceability

*An executable research notebook. Running it top-to-bottom both demonstrates the design and **materializes a `chronicle` wheel** you can install and import anywhere.*

**Author:** Sanjay Babu  
**Created:** 2026-06-28


## §1 Motivation — why span-trees are the wrong abstraction for agent traces

Current agent-observability tooling (LangSmith, LangFuse, Phoenix, AgentOps, OpenLLMetry) models traces as **hierarchical span trees** — a port of OpenTelemetry's web-request model onto LLM agents. That model has three weaknesses for research:

1. **Causality ≠ hierarchy.** When a tool result causes a reflection that    updates an earlier plan, span-trees can't express it without cycles or    duplication.
2. **Traces are mutable.** Nothing prevents a logging backend (or an attacker,    or a buggy retry) from rewriting history. There is no integrity guarantee    for audit, reproducibility, or adversarial-robustness studies.
3. **Cross-run comparison is hard.** Spans get fresh ids every run, so    *“did agent v2 produce the same intermediate reasoning as v1?”* requires    fuzzy text matching.

### The unified idea

Model an agent trace as a **content-addressed causal DAG with a semantic overlay** — borrowing Merkle-DAG ideas from git/IPFS and applying them to agent observability. Every event's id *is* the hash of its payload + its causal parents' ids.

| Concern | Falls out of the design |
|---|---|
| Causality | Edges are explicit parent pointers, not implicit nesting |
| Integrity | Content-addressing → any mutation breaks downstream hashes |
| Cross-run diff | Identical sub-traces have identical ids → set algebra works |
| Semantic search | Overlay an embedding index keyed by event id |
| Counterfactual | Forking a DAG at a node is a first-class operation |

That combined framing is the research contribution of this notebook.


In [ ]:
# Bootstrap the on-disk package layout. Subsequent cells write module
# source files into src/chronicle/ and at the end we build the wheel.
from pathlib import Path
import os, sys

ROOT = Path.cwd()
# In Jupyter the notebook's cwd is its own directory; step up one level
# so the package lands at the project root next to the notebook/ dir.
if ROOT.name == 'notebook':
    ROOT = ROOT.parent
os.chdir(ROOT)
(ROOT / 'src' / 'chronicle').mkdir(parents=True, exist_ok=True)
print('project root:', ROOT)
print('python:', sys.version.split()[0])


## §2 The `Event` type — content-addressed identity

Each event is a frozen dataclass. Its `id` is **derived**, not stored: it's the blake2b hash of a canonical-JSON encoding of `(kind, actor, payload, parents, meta)`. Two structurally-identical events get the same id — this is what enables cross-run deduplication and exact diff.

Wall-clock time is deliberately *not* part of identity. Put a timestamp in `meta` if you want it, knowing that doing so will make cross-run identity sensitive to it (which you probably don't want).


In [ ]:
_SRC = r'''from __future__ import annotations
import hashlib
import json
from dataclasses import dataclass, field
from typing import Any, Mapping


def canonical_json(obj: Any) -> bytes:
    """Stable, deterministic JSON encoding for content addressing."""
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        default=str,
    ).encode("utf-8")


def _normalize(x: Any) -> Any:
    if isinstance(x, Mapping):
        return {str(k): _normalize(v) for k, v in sorted(x.items(), key=lambda kv: str(kv[0]))}
    if isinstance(x, (list, tuple)):
        return [_normalize(v) for v in x]
    return x


@dataclass(frozen=True)
class Event:
    """A single trace event. Identity is the blake2b hash of its content + parents.

    Hashed fields: `kind`, `actor`, `payload`, `parents`, `meta`.
    Non-hashed: `telemetry` — for measurements (duration_ms, started_at_ns,
    token counts, cost, etc.) that vary across runs and would otherwise destroy
    cross-run determinism. Treat telemetry as advisory/observational data, not
    part of the trace's logical content.
    """
    kind: str
    actor: str
    payload: Mapping[str, Any] = field(default_factory=dict)
    parents: tuple[str, ...] = ()
    meta: Mapping[str, Any] = field(default_factory=dict)
    telemetry: Mapping[str, Any] = field(default_factory=dict)

    @property
    def id(self) -> str:
        body = {
            "kind": self.kind,
            "actor": self.actor,
            "payload": _normalize(self.payload),
            "parents": list(self.parents),
            "meta": _normalize(self.meta),
        }
        return hashlib.blake2b(canonical_json(body), digest_size=16).hexdigest()

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "kind": self.kind,
            "actor": self.actor,
            "payload": _normalize(self.payload),
            "parents": list(self.parents),
            "meta": _normalize(self.meta),
            "telemetry": _normalize(self.telemetry),
        }

    @classmethod
    def from_dict(cls, d: dict) -> "Event":
        return cls(
            kind=d["kind"],
            actor=d["actor"],
            payload=d.get("payload", {}),
            parents=tuple(d.get("parents", [])),
            meta=d.get("meta", {}),
            telemetry=d.get("telemetry", {}),
        )
'''
from pathlib import Path
_p = Path('src/chronicle/event.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


In [ ]:
# Sanity check: identical content → identical id; one-byte change → new id.
import importlib, sys
sys.path.insert(0, str((ROOT / 'src').resolve()))
for mod in list(sys.modules):
    if mod == 'chronicle' or mod.startswith('chronicle.'):
        del sys.modules[mod]
from chronicle.event import Event

a = Event(kind='prompt', actor='user', payload={'text': 'hello'})
b = Event(kind='prompt', actor='user', payload={'text': 'hello'})
c = Event(kind='prompt', actor='user', payload={'text': 'hellO'})
print('a.id == b.id  ->', a.id == b.id)  # True
print('a.id == c.id  ->', a.id == c.id)  # False
print('a.id =', a.id)


## §3 The `Chronicle` core — DAG construction & causal queries

`Chronicle.record(...)` appends an event whose `parents` must already exist (this guarantees acyclicity by construction — you cannot record an event with a future parent). Two adjacency-list indexes (`_events`, `_children`) support ancestor/descendant queries in linear time.


In [ ]:
_SRC = r'''from __future__ import annotations
import json
from collections import defaultdict, deque
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional

from .event import Event


class Chronicle:
    """A content-addressed causal DAG of agent events.

    Three views over one structure:
      * Causal — `ancestors`, `descendants`, `why`, `affects`
      * Integrity — `verify`, `root` (Merkle)
      * Semantic — `search` (cosine over an embedding overlay)

    Plus first-class counterfactual `branch_at` + `Chronicle.diff`.
    """

    def __init__(self, embedder: Optional[Any] = None):
        from .semantic import HashingEmbedder, SemanticIndex
        self._events: dict[str, Event] = {}
        self._children: dict[str, set[str]] = defaultdict(set)
        self._order: list[str] = []
        self._embedder = embedder if embedder is not None else HashingEmbedder()
        self._index = SemanticIndex(self._embedder)

    # ---- recording ----

    def record(
        self,
        kind: str,
        *,
        actor: str,
        payload: Optional[Mapping[str, Any]] = None,
        parents: Iterable[str] = (),
        meta: Optional[Mapping[str, Any]] = None,
        telemetry: Optional[Mapping[str, Any]] = None,
    ) -> Event:
        payload = dict(payload) if payload is not None else {}
        meta = dict(meta) if meta is not None else {}
        telemetry = dict(telemetry) if telemetry is not None else {}
        parents = tuple(parents)
        for p in parents:
            if p not in self._events:
                raise ValueError(f"parent event {p!r} not recorded")
        evt = Event(
            kind=kind, actor=actor, payload=payload, parents=parents,
            meta=meta, telemetry=telemetry,
        )
        if evt.id not in self._events:
            self._events[evt.id] = evt
            self._order.append(evt.id)
            for p in parents:
                self._children[p].add(evt.id)
            self._index.add(evt)
        return evt

    # ---- telemetry rollups ----

    def stats(self, prices: Optional[Mapping[str, tuple]] = None) -> dict:
        from .telemetry import summarize
        return summarize(self, prices=prices)

    def timeline(self) -> list[dict]:
        """Flat list of events with their telemetry, in recording order."""
        return [
            {
                "id": e.id,
                "kind": e.kind,
                "actor": e.actor,
                "payload": dict(e.payload),
                "telemetry": dict(e.telemetry),
                "parents": list(e.parents),
            }
            for e in self
        ]

    # ---- accessors ----

    def __len__(self) -> int:
        return len(self._events)

    def __iter__(self):
        return iter(self._events[i] for i in self._order)

    def __contains__(self, eid: str) -> bool:
        return eid in self._events

    @property
    def events(self) -> dict[str, Event]:
        return dict(self._events)

    @property
    def order(self) -> list[str]:
        return list(self._order)

    # ---- causal queries ----

    def ancestors(self, eid: str) -> set[str]:
        seen: set[str] = set()
        q = deque([eid])
        while q:
            x = q.popleft()
            for p in self._events[x].parents:
                if p not in seen:
                    seen.add(p)
                    q.append(p)
        return seen

    def descendants(self, eid: str) -> set[str]:
        seen: set[str] = set()
        q = deque([eid])
        while q:
            x = q.popleft()
            for c in self._children.get(x, ()):
                if c not in seen:
                    seen.add(c)
                    q.append(c)
        return seen

    def why(self, eid: str) -> list[Event]:
        anc = self.ancestors(eid) | {eid}
        return [self._events[i] for i in self._order if i in anc]

    def affects(self, eid: str) -> list[Event]:
        desc = self.descendants(eid) | {eid}
        return [self._events[i] for i in self._order if i in desc]

    def lineage(self, eid: str) -> list[Event]:
        """A linear chain via first-parent. Useful for printing."""
        out: list[Event] = []
        cur = self._events[eid]
        while True:
            out.append(cur)
            if not cur.parents:
                break
            cur = self._events[cur.parents[0]]
        return list(reversed(out))

    # ---- semantic ----

    def search(self, query: str, k: int = 5) -> list[tuple[Event, float]]:
        hits = self._index.search(query, k)
        return [(self._events[eid], score) for eid, score in hits]

    # ---- integrity ----

    def verify(self) -> tuple[bool, list[str]]:
        from .integrity import verify as _v
        return _v(self)

    def root(self) -> str:
        from .integrity import root as _r
        return _r(self)

    # ---- branching ----

    def branch_at(self, eid: str) -> "Chronicle":
        from .branch import branch_at as _b
        return _b(self, eid)

    @staticmethod
    def diff(a: "Chronicle", b: "Chronicle") -> dict:
        from .branch import diff as _d
        return _d(a, b)

    # ---- persistence ----

    def dump(self, path) -> None:
        p = Path(path)
        with p.open("w", encoding="utf-8") as f:
            for eid in self._order:
                f.write(json.dumps(self._events[eid].to_dict(), ensure_ascii=False) + "\n")

    @classmethod
    def load(cls, path, embedder=None) -> "Chronicle":
        c = cls(embedder=embedder)
        with Path(path).open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                evt = Event.from_dict(d)
                if evt.id != d["id"]:
                    raise ValueError(
                        f"corrupt trace at line: stored id={d['id']!r} but content hashes to {evt.id!r}"
                    )
                c._events[evt.id] = evt
                c._order.append(evt.id)
                for p in evt.parents:
                    c._children[p].add(evt.id)
                c._index.add(evt)
        return c
'''
from pathlib import Path
_p = Path('src/chronicle/core.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §4 Integrity — Merkle root & `verify()`

`verify()` re-computes every event's id from its content and checks it against the key it's stored under. Because parents are part of the hashed body, tampering with *any* event invalidates not just that event's id but every descendant that referenced it — exactly the Merkle property.

`root()` computes a Merkle root over the **set** of event ids (sorted, hashed pairwise). It identifies the set of events, not their recording order — order is recoverable from the parent edges and is irrelevant to what was recorded.


In [ ]:
_SRC = r'''from __future__ import annotations
import hashlib
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from .core import Chronicle


def merkle_root(ids: list[str]) -> str:
    """Compute a Merkle root over a list of hex event ids.

    Sorted to be order-independent — the root identifies the *set* of events,
    not the recording order. (Order is recoverable from the parent edges.)
    """
    if not ids:
        return hashlib.blake2b(b"", digest_size=16).hexdigest()
    layer = [bytes.fromhex(i) for i in sorted(set(ids))]
    while len(layer) > 1:
        nxt = []
        for i in range(0, len(layer), 2):
            a = layer[i]
            b = layer[i + 1] if i + 1 < len(layer) else layer[i]
            nxt.append(hashlib.blake2b(a + b, digest_size=16).digest())
        layer = nxt
    return layer[0].hex()


def verify(chronicle: "Chronicle") -> tuple[bool, list[str]]:
    """Return (ok, bad_ids). bad_ids contains stored keys whose value's content
    no longer hashes to the key, or events whose parents are missing.
    """
    bad: list[str] = []
    for stored_id, evt in chronicle._events.items():
        if evt.id != stored_id:
            bad.append(stored_id)
            continue
        for p in evt.parents:
            if p not in chronicle._events:
                bad.append(stored_id)
                break
    return (not bad, bad)


def root(chronicle: "Chronicle") -> str:
    return merkle_root(list(chronicle._events.keys()))
'''
from pathlib import Path
_p = Path('src/chronicle/integrity.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §5 Semantic overlay — search by meaning, not by id

Each recorded event is embedded into a fixed-dim vector and added to an in-memory cosine index. The default `HashingEmbedder` uses character trigrams + the hashing trick — no external model required, so the wheel's only runtime dep is `numpy`. A `SentenceTransformerEmbedder` adapter is included for higher semantic quality when you can afford the dependency.


In [ ]:
_SRC = r'''from __future__ import annotations
import hashlib
from typing import Any, TYPE_CHECKING

import numpy as np

if TYPE_CHECKING:
    from .event import Event


class HashingEmbedder:
    """Char-trigram hashing-trick embedder. Stdlib + numpy only.

    Cheap, deterministic, and good enough for similarity-based retrieval over
    short agent events. Swap in `SentenceTransformerEmbedder` for higher
    semantic quality at the cost of an extra dependency + model download.
    """

    def __init__(self, dim: int = 512, ngram: int = 3):
        self.dim = dim
        self.ngram = ngram

    @staticmethod
    def _event_text(evt: "Event") -> str:
        from .event import canonical_json
        return " ".join([
            evt.kind,
            evt.actor,
            canonical_json(dict(evt.payload)).decode("utf-8", errors="replace"),
        ]).lower()

    def embed(self, text: str) -> np.ndarray:
        vec = np.zeros(self.dim, dtype=np.float32)
        s = text.lower()
        if len(s) < self.ngram:
            s = s.ljust(self.ngram)
        for i in range(len(s) - self.ngram + 1):
            tri = s[i:i + self.ngram]
            h = int.from_bytes(
                hashlib.blake2b(tri.encode("utf-8"), digest_size=4).digest(),
                "big",
            )
            idx = h % self.dim
            sign = 1.0 if (h >> 16) & 1 else -1.0
            vec[idx] += sign
        n = float(np.linalg.norm(vec))
        if n > 0:
            vec /= n
        return vec

    def embed_event(self, evt: "Event") -> np.ndarray:
        return self.embed(self._event_text(evt))


class SentenceTransformerEmbedder:
    """Optional adapter. Requires `pip install chronicle[st]`."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer  # type: ignore
        self._m = SentenceTransformer(model_name)
        self.dim = int(self._m.get_sentence_embedding_dimension())

    @staticmethod
    def _event_text(evt: "Event") -> str:
        return f"{evt.kind} {evt.actor} {evt.payload}"

    def embed(self, text: str) -> np.ndarray:
        v = self._m.encode([text], normalize_embeddings=True)[0]
        return np.asarray(v, dtype=np.float32)

    def embed_event(self, evt: "Event") -> np.ndarray:
        return self.embed(self._event_text(evt))


class SemanticIndex:
    """Cosine-similarity index over an embedder. L2-normalized vectors so the
    dot product equals cosine similarity.
    """

    def __init__(self, embedder: Any):
        self.embedder = embedder
        self._ids: list[str] = []
        self._mat: np.ndarray | None = None

    def add(self, evt: "Event") -> None:
        v = self.embedder.embed_event(evt).astype(np.float32)[None, :]
        if self._mat is None:
            self._mat = v
        else:
            self._mat = np.vstack([self._mat, v])
        self._ids.append(evt.id)

    def search(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        if self._mat is None or not self._ids:
            return []
        q = self.embedder.embed(query).astype(np.float32)
        sims = self._mat @ q
        k = min(k, len(self._ids))
        order = np.argpartition(-sims, k - 1)[:k]
        order = order[np.argsort(-sims[order])]
        return [(self._ids[int(i)], float(sims[int(i)])) for i in order]
'''
from pathlib import Path
_p = Path('src/chronicle/semantic.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §6 Ergonomic API — `@tool` and `span(...)`

Threading parent-ids through every call is tedious. Two helpers fix this:

- `chronicle.span(c, actor=..., parent=...)` — context manager. Inside, every `s.thought(...)` / `s.answer(...)` chains automatically from the previous emit via a `contextvar`.
- `@chronicle.tool` — decorator. When the wrapped function is called inside an active span, it records a `tool_call` before and a `tool_result` after, chained from the current contextvar parent, and advances the parent so subsequent emits chain from the result.


In [ ]:
_SRC = r'''from __future__ import annotations
import contextvars
import functools
import json
from contextlib import contextmanager
from typing import Any, Callable, Optional, TYPE_CHECKING

if TYPE_CHECKING:
    from .core import Chronicle

_current_parent: contextvars.ContextVar[Optional[str]] = contextvars.ContextVar(
    "chronicle_current_parent", default=None
)
_current_chronicle: contextvars.ContextVar[Optional["Chronicle"]] = contextvars.ContextVar(
    "chronicle_current", default=None
)


def _json_safe(x: Any) -> Any:
    try:
        json.dumps(x)
        return x
    except Exception:
        return repr(x)


class Span:
    """A helper inside `span(...)`. Each emit advances the contextvar parent so
    subsequent emits (and any `@tool`-decorated calls) chain automatically.
    """

    def __init__(self, chronicle: "Chronicle", actor: str):
        self._c = chronicle
        self._actor = actor

    def _emit(self, kind: str, payload: dict, meta: Optional[dict] = None) -> str:
        parent = _current_parent.get()
        parents = (parent,) if parent else ()
        evt = self._c.record(
            kind, actor=self._actor, payload=payload, parents=parents, meta=meta or {}
        )
        _current_parent.set(evt.id)
        return evt.id

    def thought(self, text: str, **meta) -> str:
        return self._emit("thought", {"text": text}, meta)

    def answer(self, text: str, **meta) -> str:
        return self._emit("answer", {"text": text}, meta)

    def emit(self, kind: str, payload: dict, **meta) -> str:
        return self._emit(kind, payload, meta)


@contextmanager
def span(chronicle: "Chronicle", actor: str, parent: Optional[str] = None):
    """Context manager: events emitted inside (via `Span` or `@tool`) chain
    from `parent` automatically via a contextvar.
    """
    if parent is None:
        parent = _current_parent.get()
    sp = Span(chronicle, actor)
    tok_c = _current_chronicle.set(chronicle)
    tok_p = _current_parent.set(parent)
    try:
        yield sp
    finally:
        _current_parent.reset(tok_p)
        _current_chronicle.reset(tok_c)


@contextmanager
def active(chronicle: "Chronicle", parent: Optional[str] = None):
    """Make `chronicle` the active one for `@tool`-decorated calls without
    introducing an actor/span. Useful for top-level scripts.
    """
    tok_c = _current_chronicle.set(chronicle)
    tok_p = _current_parent.set(parent)
    try:
        yield chronicle
    finally:
        _current_parent.reset(tok_p)
        _current_chronicle.reset(tok_c)


def tool(name: Optional[str] = None) -> Callable:
    """Decorator. When the wrapped function is called inside an active
    Chronicle, records a `tool_call` before and a `tool_result` after, chained
    from the current contextvar parent. Wall-clock duration is captured in
    `telemetry` (non-hashed) so durations don't break cross-run identity.
    """
    import time

    def _decorate(fn: Callable) -> Callable:
        tool_name = name or fn.__name__

        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            c = _current_chronicle.get()
            if c is None:
                return fn(*args, **kwargs)
            parent = _current_parent.get()
            parents = (parent,) if parent else ()
            started = time.time_ns()
            t0 = time.perf_counter_ns()
            call = c.record(
                "tool_call",
                actor=f"tool:{tool_name}",
                payload={
                    "name": tool_name,
                    "args": [_json_safe(a) for a in args],
                    "kwargs": {k: _json_safe(v) for k, v in kwargs.items()},
                },
                parents=parents,
                telemetry={"started_at_ns": started},
            )
            _current_parent.set(call.id)
            try:
                out = fn(*args, **kwargs)
            except Exception as e:
                dt_ms = (time.perf_counter_ns() - t0) / 1e6
                err = c.record(
                    "error",
                    actor=f"tool:{tool_name}",
                    payload={"name": tool_name, "error": repr(e)},
                    parents=(call.id,),
                    telemetry={"duration_ms": dt_ms},
                )
                _current_parent.set(err.id)
                raise
            dt_ms = (time.perf_counter_ns() - t0) / 1e6
            res = c.record(
                "tool_result",
                actor=f"tool:{tool_name}",
                payload={"name": tool_name, "result": _json_safe(out)},
                parents=(call.id,),
                telemetry={"duration_ms": dt_ms},
            )
            _current_parent.set(res.id)
            return out

        return wrapper

    # Allow @tool as well as @tool("custom_name")
    if callable(name):
        fn, name = name, None
        return _decorate(fn)
    return _decorate
'''
from pathlib import Path
_p = Path('src/chronicle/ergonomics.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §7 Counterfactual branching — the operation that justifies content-addressing

Because ancestors are shared *by id* (not by reference), forking a Chronicle at any event is essentially free: just copy the ancestor set into a new container. `Chronicle.diff(orig, branch)` is then exact set algebra over event ids — no fuzzy text matching, no heuristics.


In [ ]:
_SRC = r'''from __future__ import annotations
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from .core import Chronicle


def branch_at(chronicle: "Chronicle", eid: str) -> "Chronicle":
    """Return a new Chronicle containing `eid` and all its causal ancestors.

    Because events are content-addressed, the new Chronicle shares ids with the
    original — `Chronicle.diff(orig, branch)` is exact set algebra, not fuzzy
    matching. The caller can record() additional events on top of the branch.
    """
    from .core import Chronicle
    if eid not in chronicle._events:
        raise KeyError(eid)
    keep = chronicle.ancestors(eid) | {eid}
    new = Chronicle(embedder=chronicle._embedder)
    for i in chronicle._order:
        if i in keep:
            evt = chronicle._events[i]
            new._events[evt.id] = evt
            new._order.append(evt.id)
            for p in evt.parents:
                new._children[p].add(evt.id)
            new._index.add(evt)
    return new


def diff(a: "Chronicle", b: "Chronicle") -> dict:
    """Exact set difference over event ids."""
    aids = set(a._events.keys())
    bids = set(b._events.keys())
    return {
        "only_in_a": sorted(aids - bids),
        "only_in_b": sorted(bids - aids),
        "common": sorted(aids & bids),
    }
'''
from pathlib import Path
_p = Path('src/chronicle/branch.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §8 Telemetry — LLM calls, tokens, durations, cost

Observability that's useful in practice means recording **what models ran, how long they took, how many tokens they consumed, and what they cost** — alongside per-tool latencies. Two design points worth flagging:

1. **Telemetry lives in a separate `Event.telemetry` field that is NOT    part of the content hash.** Durations and timestamps vary between runs;    if they were hashed, the headline cross-run-identity property would    collapse. Keeping telemetry outside the hash means we can record    wall-clock data freely without sacrificing determinism.
2. **`@llm` works the same way `@tool` does**, but extracts token usage    from the response object (Anthropic `usage.input_tokens` / OpenAI    `usage.prompt_tokens` are both recognized) and attaches an estimated    USD cost from a per-model price table.

`Chronicle.stats()` rolls everything up: event-kind histogram, total tokens, per-model breakdown, total cost.


In [ ]:
_SRC = r'''"""LLM-call instrumentation, duration tracking, and run-level aggregations.

Telemetry data (durations, token counts, cost, wall-clock timestamps) lives in
`Event.telemetry` — a field deliberately *excluded* from the content hash so
recording it does not destroy the cross-run determinism property.
"""
from __future__ import annotations
import functools
import time
from collections import defaultdict
from typing import Any, Callable, Iterable, Mapping, Optional, TYPE_CHECKING

from .ergonomics import _current_chronicle, _current_parent, _json_safe

if TYPE_CHECKING:
    from .core import Chronicle


# ---- approximate USD price per 1M tokens (input, output) ---------------------
# Snapshot for illustrative dashboard cost estimates. Override via
# `chronicle.set_prices({...})` or pass `prices=` to `Chronicle.stats(...)`.
DEFAULT_PRICES: dict[str, tuple[float, float]] = {
    "claude-opus-4-7": (15.0, 75.0),
    "claude-opus-4-6": (15.0, 75.0),
    "claude-sonnet-4-6": (3.0, 15.0),
    "claude-haiku-4-5": (1.0, 5.0),
    "gpt-4o": (5.0, 15.0),
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4-turbo": (10.0, 30.0),
    "o1": (15.0, 60.0),
}

_PRICES = dict(DEFAULT_PRICES)


def set_prices(prices: Mapping[str, tuple[float, float]]) -> None:
    """Override the default price table at module level."""
    global _PRICES
    _PRICES = dict(prices)


def estimate_cost_usd(model: str, input_tokens: int, output_tokens: int,
                      prices: Optional[Mapping[str, tuple]] = None) -> float:
    pt = prices if prices is not None else _PRICES
    p_in, p_out = pt.get(model, (0.0, 0.0))
    return (input_tokens * p_in + output_tokens * p_out) / 1_000_000.0


# ---- response sniffing -------------------------------------------------------

def _extract_usage(response: Any) -> dict:
    """Best-effort token-count extraction. Recognizes Anthropic (`input_tokens`/
    `output_tokens`) and OpenAI (`prompt_tokens`/`completion_tokens`) response
    shapes, plain dicts with the same keys, and anything `usage`-shaped.
    """
    out = {"input_tokens": 0, "output_tokens": 0}
    u = None
    if hasattr(response, "usage"):
        u = response.usage
    elif isinstance(response, Mapping) and "usage" in response:
        u = response["usage"]
    if u is None:
        return out

    def _get(obj, key):
        if isinstance(obj, Mapping):
            return obj.get(key)
        return getattr(obj, key, None)

    for src, dst in [
        ("input_tokens", "input_tokens"),
        ("prompt_tokens", "input_tokens"),
        ("output_tokens", "output_tokens"),
        ("completion_tokens", "output_tokens"),
    ]:
        v = _get(u, src)
        if v is not None:
            out[dst] = int(v)
    out["total_tokens"] = out["input_tokens"] + out["output_tokens"]
    return out


def _extract_text(response: Any) -> str:
    """Best-effort text extraction from an Anthropic or OpenAI response."""
    if hasattr(response, "content"):
        try:
            c = response.content
            if isinstance(c, list) and c:
                first = c[0]
                return getattr(first, "text", None) or (first.get("text") if isinstance(first, Mapping) else None) or str(c)
            return str(c)
        except Exception:
            pass
    if hasattr(response, "choices"):
        try:
            ch = response.choices[0]
            msg = getattr(ch, "message", None) or (ch.get("message") if isinstance(ch, Mapping) else None)
            if msg is not None:
                return getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, Mapping) else None) or str(msg)
        except Exception:
            pass
    if isinstance(response, Mapping):
        for k in ("text", "content", "output_text"):
            if k in response:
                return str(response[k])
    return str(response)


def _extract_model(response: Any, default: Optional[str]) -> str:
    for src in ("model", "_model"):
        v = getattr(response, src, None)
        if v:
            return str(v)
        if isinstance(response, Mapping) and src in response:
            return str(response[src])
    return default or "unknown"


# ---- @llm decorator ----------------------------------------------------------

def llm(name_or_fn=None, *, model: Optional[str] = None) -> Callable:
    """Decorator for functions that call an LLM. When invoked inside an active
    Chronicle, records `llm_call` then `llm_result`, with token counts, model,
    and wall-clock duration in `telemetry`.

    Usage::

        @llm(model="claude-opus-4-7")
        def ask(messages):
            return client.messages.create(model="claude-opus-4-7", messages=messages)

        # or just @llm — model is read from the response
        @llm
        def ask(messages): ...
    """

    def _decorate(fn: Callable) -> Callable:
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            c = _current_chronicle.get()
            if c is None:
                return fn(*args, **kwargs)
            parent = _current_parent.get()
            parents = (parent,) if parent else ()
            started = time.time_ns()
            t0 = time.perf_counter_ns()
            call = c.record(
                "llm_call",
                actor=f"llm:{model or fn.__name__}",
                payload={
                    "model": model,
                    "args": [_json_safe(a) for a in args],
                    "kwargs": {k: _json_safe(v) for k, v in kwargs.items()},
                },
                parents=parents,
                telemetry={"started_at_ns": started},
            )
            _current_parent.set(call.id)
            try:
                result = fn(*args, **kwargs)
            except Exception as e:
                dt_ms = (time.perf_counter_ns() - t0) / 1e6
                err = c.record(
                    "error",
                    actor=f"llm:{model or fn.__name__}",
                    payload={"error": repr(e)},
                    parents=(call.id,),
                    telemetry={"duration_ms": dt_ms},
                )
                _current_parent.set(err.id)
                raise
            dt_ms = (time.perf_counter_ns() - t0) / 1e6
            usage = _extract_usage(result)
            text = _extract_text(result)
            actual_model = _extract_model(result, model)
            res = c.record(
                "llm_result",
                actor=f"llm:{actual_model}",
                payload={
                    "model": actual_model,
                    "text": text,
                    "input_tokens": usage["input_tokens"],
                    "output_tokens": usage["output_tokens"],
                    "total_tokens": usage["total_tokens"],
                },
                parents=(call.id,),
                telemetry={
                    "duration_ms": dt_ms,
                    "cost_usd": estimate_cost_usd(actual_model, usage["input_tokens"], usage["output_tokens"]),
                },
            )
            _current_parent.set(res.id)
            return result

        return wrapper

    # Support @llm and @llm(model="...")
    if callable(name_or_fn):
        return _decorate(name_or_fn)
    return _decorate


# ---- manual recording API ----------------------------------------------------

def record_llm(
    chronicle: "Chronicle",
    *,
    model: str,
    prompt: Any,
    response: Any,
    input_tokens: int = 0,
    output_tokens: int = 0,
    duration_ms: float = 0.0,
    parents: Iterable[str] = (),
    extra_telemetry: Optional[Mapping[str, Any]] = None,
) -> tuple:
    """Manual API for recording an LLM call when you can't or don't want to use
    the `@llm` decorator. Returns (call_event, result_event).
    """
    parents = tuple(parents)
    call = chronicle.record(
        "llm_call",
        actor=f"llm:{model}",
        payload={"model": model, "prompt": _json_safe(prompt)},
        parents=parents,
    )
    tele = {"duration_ms": float(duration_ms),
            "cost_usd": estimate_cost_usd(model, input_tokens, output_tokens)}
    if extra_telemetry:
        tele.update(extra_telemetry)
    res = chronicle.record(
        "llm_result",
        actor=f"llm:{model}",
        payload={
            "model": model,
            "text": _extract_text(response) if not isinstance(response, str) else response,
            "input_tokens": int(input_tokens),
            "output_tokens": int(output_tokens),
            "total_tokens": int(input_tokens) + int(output_tokens),
        },
        parents=(call.id,),
        telemetry=tele,
    )
    return call, res


# ---- run summaries -----------------------------------------------------------

def summarize(chronicle: "Chronicle", prices: Optional[Mapping[str, tuple]] = None) -> dict:
    """Roll up per-run telemetry. Returns counts, total tokens, latency, cost,
    and a per-model breakdown.
    """
    by_kind: dict[str, int] = defaultdict(int)
    by_actor: dict[str, int] = defaultdict(int)
    by_model: dict[str, dict] = defaultdict(lambda: {
        "calls": 0, "input_tokens": 0, "output_tokens": 0,
        "duration_ms": 0.0, "cost_usd": 0.0,
    })
    llm_calls = 0
    tool_calls = 0
    in_tok = out_tok = 0
    llm_ms = 0.0
    tool_ms = 0.0
    total_cost = 0.0

    for evt in chronicle:
        by_kind[evt.kind] += 1
        by_actor[evt.actor] += 1
        if evt.kind == "llm_result":
            llm_calls += 1
            it = int(evt.payload.get("input_tokens", 0))
            ot = int(evt.payload.get("output_tokens", 0))
            in_tok += it
            out_tok += ot
            dur = float(evt.telemetry.get("duration_ms", 0.0))
            llm_ms += dur
            m = str(evt.payload.get("model", "unknown"))
            cost = (
                estimate_cost_usd(m, it, ot, prices)
                if prices is not None
                else float(evt.telemetry.get("cost_usd", estimate_cost_usd(m, it, ot)))
            )
            total_cost += cost
            row = by_model[m]
            row["calls"] += 1
            row["input_tokens"] += it
            row["output_tokens"] += ot
            row["duration_ms"] += dur
            row["cost_usd"] += cost
        elif evt.kind == "tool_result":
            tool_calls += 1
            tool_ms += float(evt.telemetry.get("duration_ms", 0.0))

    return {
        "total_events": len(chronicle),
        "by_kind": dict(by_kind),
        "by_actor": dict(by_actor),
        "llm_calls": llm_calls,
        "tool_calls": tool_calls,
        "input_tokens": in_tok,
        "output_tokens": out_tok,
        "total_tokens": in_tok + out_tok,
        "llm_duration_ms": llm_ms,
        "tool_duration_ms": tool_ms,
        "estimated_cost_usd": total_cost,
        "by_model": {k: dict(v) for k, v in by_model.items()},
    }
'''
from pathlib import Path
_p = Path('src/chronicle/telemetry.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §9 Dashboard module — a Streamlit frontend

A single-file Streamlit app, shipped inside the wheel and exposed via the console script **`chronicle-dashboard`**. It loads any JSONL trace and shows: top-level metric cards (events, LLM calls, tool calls, tokens, latency, estimated cost), event-kind histogram, per-model breakdown, sortable LLM/tool tables, the full causal DAG, semantic search, raw event drill-down, and a live integrity badge.

Streamlit, pandas, matplotlib, networkx are **optional** — declared in the `[dashboard]` extra so the core wheel stays numpy-only.


In [ ]:
_SRC = r'''"""Streamlit dashboard for chronicle traces.

Launch::

    chronicle-dashboard path/to/trace.jsonl

or programmatically::

    from chronicle.dashboard import launch
    launch("path/to/trace.jsonl")

Or from any Chronicle::

    c.dump("trace.jsonl"); launch("trace.jsonl")
"""
from __future__ import annotations
import argparse
import subprocess
import sys
from pathlib import Path
from typing import Optional


def launch(trace_path: Optional[str] = None) -> int:
    """Spawn `streamlit run` on this module."""
    here = Path(__file__).resolve()
    cmd = [sys.executable, "-m", "streamlit", "run", str(here)]
    if trace_path:
        cmd += ["--", "--trace", str(trace_path)]
    return subprocess.call(cmd)


def cli() -> None:
    """Console entry-point. Accepts an optional positional trace path."""
    p = argparse.ArgumentParser(description="Launch the chronicle dashboard.")
    p.add_argument("trace", nargs="?", default=None, help="Path to a JSONL trace.")
    args = p.parse_args()
    raise SystemExit(launch(args.trace))


def _format_dur(ms: float) -> str:
    if ms < 1000:
        return f"{ms:.0f} ms"
    if ms < 60_000:
        return f"{ms/1000:.2f} s"
    return f"{ms/60_000:.2f} min"


def _run_streamlit_app() -> None:
    """The Streamlit page itself. Only imports streamlit lazily so the rest of
    the package remains import-light.
    """
    import streamlit as st
    import pandas as pd
    import numpy as np

    p = argparse.ArgumentParser()
    p.add_argument("--trace", default=None)
    args, _ = p.parse_known_args()

    st.set_page_config(page_title="chronicle", layout="wide")
    st.title("chronicle - agent trace dashboard")
    st.caption("Merkle-DAG agent traceability  |  causal . integrity . semantic")

    from chronicle import Chronicle

    if args.trace and Path(args.trace).exists():
        trace_path = args.trace
    else:
        st.sidebar.subheader("Load trace")
        uploaded = st.sidebar.file_uploader("Upload a JSONL trace", type=["jsonl", "json"])
        if uploaded is None:
            st.info("No trace loaded. Provide --trace or upload a JSONL file in the sidebar.")
            return
        import tempfile
        tf = tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False)
        tf.write(uploaded.read())
        tf.close()
        trace_path = tf.name

    c = Chronicle.load(trace_path)
    stats = c.stats()
    ok, bad = c.verify()

    # ---- top metric cards ----
    cols = st.columns(6)
    cols[0].metric("Events", stats["total_events"])
    cols[1].metric("LLM calls", stats["llm_calls"])
    cols[2].metric("Tool calls", stats["tool_calls"])
    cols[3].metric("Tokens", f'{stats["total_tokens"]:,}')
    cols[4].metric("LLM latency", _format_dur(stats["llm_duration_ms"]))
    cols[5].metric("Est. cost", f'${stats["estimated_cost_usd"]:.4f}')

    if ok:
        st.success(f"Integrity OK . Merkle root: `{c.root()}`")
    else:
        st.error(f"INTEGRITY FAILED . Bad event ids: {bad}")

    tabs = st.tabs(["Overview", "LLM calls", "Tool calls", "DAG", "Search", "Raw"])

    # ---- Overview ----
    with tabs[0]:
        st.subheader("Event mix")
        by_kind = stats["by_kind"]
        if by_kind:
            df = pd.DataFrame({"kind": list(by_kind), "count": list(by_kind.values())})
            st.bar_chart(df.set_index("kind"))

        if stats["by_model"]:
            st.subheader("Per-model breakdown")
            rows = []
            for model, row in stats["by_model"].items():
                rows.append({
                    "model": model,
                    "calls": row["calls"],
                    "input_tokens": row["input_tokens"],
                    "output_tokens": row["output_tokens"],
                    "duration_ms": row["duration_ms"],
                    "cost_usd": row["cost_usd"],
                })
            st.dataframe(pd.DataFrame(rows), use_container_width=True)

    # ---- LLM calls ----
    with tabs[1]:
        rows = []
        evts = list(c)
        by_id = {e.id: e for e in evts}
        for e in evts:
            if e.kind == "llm_result":
                call = by_id.get(e.parents[0]) if e.parents else None
                rows.append({
                    "id": e.id[:10],
                    "model": e.payload.get("model"),
                    "input_tokens": e.payload.get("input_tokens", 0),
                    "output_tokens": e.payload.get("output_tokens", 0),
                    "duration_ms": e.telemetry.get("duration_ms", 0.0),
                    "cost_usd": e.telemetry.get("cost_usd", 0.0),
                    "text": (e.payload.get("text") or "")[:140],
                    "call_args": (str(call.payload.get("args")) if call else ""),
                })
        if rows:
            df = pd.DataFrame(rows).sort_values("duration_ms", ascending=False)
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            st.info("No llm_result events in this trace.")

    # ---- Tool calls ----
    with tabs[2]:
        rows = []
        for e in c:
            if e.kind == "tool_result":
                rows.append({
                    "id": e.id[:10],
                    "tool": e.payload.get("name"),
                    "duration_ms": e.telemetry.get("duration_ms", 0.0),
                    "result": str(e.payload.get("result"))[:160],
                })
        if rows:
            df = pd.DataFrame(rows).sort_values("duration_ms", ascending=False)
            st.dataframe(df, use_container_width=True, hide_index=True)
            st.subheader("Tool latency distribution")
            st.bar_chart(df.set_index("tool")[["duration_ms"]])
        else:
            st.info("No tool_result events in this trace.")

    # ---- DAG ----
    with tabs[3]:
        import io
        import matplotlib.pyplot as plt
        import networkx as nx
        g = nx.DiGraph()
        for e in c:
            g.add_node(e.id[:6], label=f"{e.kind}\n{e.actor[:20]}")
            for parent in e.parents:
                g.add_edge(parent[:6], e.id[:6])
        try:
            pos = nx.nx_agraph.graphviz_layout(g, prog="dot")
        except Exception:
            pos = nx.spring_layout(g, seed=42)

        fig, ax = plt.subplots(figsize=(12, max(4, len(g) * 0.4)))
        nx.draw_networkx_nodes(g, pos, ax=ax, node_size=1800, node_color="#cfe2ff", edgecolors="#1f4e79")
        nx.draw_networkx_edges(g, pos, ax=ax, arrowsize=14, edge_color="#666")
        labels = {n: g.nodes[n].get("label", n) for n in g.nodes}
        nx.draw_networkx_labels(g, pos, labels=labels, ax=ax, font_size=7)
        ax.axis("off")
        st.pyplot(fig)

    # ---- Search ----
    with tabs[4]:
        q = st.text_input("Semantic search", value="")
        if q:
            hits = c.search(q, k=10)
            for evt, score in hits:
                with st.container(border=True):
                    st.markdown(f"**{evt.kind}** . `{evt.actor}` . score `{score:+.3f}`")
                    st.code(str(evt.payload)[:600], language="json")

    # ---- Raw ----
    with tabs[5]:
        for e in c:
            with st.expander(f"{e.kind} | {e.actor} | {e.id[:12]}"):
                st.json({"payload": dict(e.payload), "telemetry": dict(e.telemetry),
                         "meta": dict(e.meta), "parents": list(e.parents)})


# Streamlit runs this module as __main__ when invoked via `streamlit run`.
if __name__ == "__main__":
    _run_streamlit_app()
'''
from pathlib import Path
_p = Path('src/chronicle/dashboard.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §10 Public API & `chronicle.demo()`

The package's `__init__.py` re-exports the public surface and defines `demo()` — a self-contained toy agent that includes a mock LLM call so the trace has both `tool_*` and `llm_*` events to feed the dashboard.


In [ ]:
_SRC = r'''"""chronicle — Merkle-DAG agent traceability.

A content-addressed causal DAG of agent events, with integrity verification
and a semantic search overlay. Built as a research artifact; see the notebook
for motivation and design.
"""
from .event import Event, canonical_json
from .core import Chronicle
from .semantic import HashingEmbedder, SentenceTransformerEmbedder, SemanticIndex
from .integrity import verify, merkle_root, root
from .ergonomics import span, tool, active
from .branch import branch_at, diff
from .telemetry import (
    llm,
    record_llm,
    summarize,
    estimate_cost_usd,
    set_prices,
    DEFAULT_PRICES,
)

__version__ = "0.2.0"

__all__ = [
    "Chronicle",
    "Event",
    "canonical_json",
    "HashingEmbedder",
    "SentenceTransformerEmbedder",
    "SemanticIndex",
    "verify",
    "merkle_root",
    "root",
    "span",
    "tool",
    "active",
    "branch_at",
    "diff",
    "llm",
    "record_llm",
    "summarize",
    "estimate_cost_usd",
    "set_prices",
    "DEFAULT_PRICES",
    "demo",
    "__version__",
]


def demo() -> "Chronicle":
    """Run a small end-to-end toy agent and return its Chronicle.

    Includes a mock LLM call so the trace exhibits both `tool_*` and `llm_*`
    events — useful for exercising the dashboard. No real API key required.
    """
    import types

    class _MockLLM:
        """Deterministic fake LLM client. Returns Anthropic-shaped responses."""
        def __init__(self, model: str = "claude-opus-4-7"):
            self.model = model

        def respond(self, prompt: str):
            ip = max(1, len(prompt) // 4)
            op = 24
            return types.SimpleNamespace(
                model=self.model,
                usage=types.SimpleNamespace(input_tokens=ip, output_tokens=op),
                content=[types.SimpleNamespace(text=f"(mock answer) {prompt[:60]}")],
            )

    c = Chronicle()
    prompt = c.record(
        "prompt",
        actor="user",
        payload={"text": "find me a vegan lasagna recipe"},
    )

    client = _MockLLM("claude-opus-4-7")

    @tool("recipe_search")
    def recipe_search(query: str) -> list:
        corpus = {
            "vegan lasagna": ["tofu ricotta", "cashew bechamel", "spinach", "zucchini"],
            "chicken curry": ["onions", "garlic", "garam masala"],
        }
        for k, v in corpus.items():
            if all(w in k for w in query.lower().split()):
                return v
        return []

    @llm(model="claude-opus-4-7")
    def ask(text: str):
        return client.respond(text)

    with span(c, actor="agent:planner", parent=prompt.id) as s:
        s.thought("the user wants a vegan lasagna recipe; search then summarize")
        hits = recipe_search("vegan lasagna")
        s.thought(f"found {len(hits)} ingredients; asking the LLM to compose")
        ask(f"Compose a short vegan lasagna recipe using: {', '.join(hits)}")
        s.answer(f"Here's a vegan lasagna using: {', '.join(hits)}")

    return c
'''
from pathlib import Path
_p = Path('src/chronicle/__init__.py'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


## §11 End-to-end demo — recording, querying, verifying, branching, telemetry

We exercise every headline property *before* building the wheel, so this notebook stands as the verification suite. Seven checks:

1. Recording chains correctly via the ergonomic API.
2. **Cross-run identity**: two independent runs produce the same Merkle root    (even though durations differ — telemetry is excluded from the hash).
3. **Tamper detection**: a manual mutation is caught by `verify()`.
4. **Causal query**: `why()` returns exactly the causal sub-DAG.
5. **Semantic recall**: `search()` finds a planted query by meaning.
6. **Counterfactual diff**: a branched continuation diffs to exactly the divergent events.
7. **Telemetry rollup**: `stats()` reports tokens, durations, and per-model cost.


In [ ]:
# Reload the package source from disk (the writefile cells above wrote it)
import importlib, sys
for mod in list(sys.modules):
    if mod == 'chronicle' or mod.startswith('chronicle.'):
        del sys.modules[mod]
import chronicle
from chronicle import Chronicle, span, tool, llm
print('loaded chronicle from:', chronicle.__file__)


In [ ]:
# --- toy agent with a mock LLM (no API key needed) ------------------------
import types

class MockLLM:
    '''Deterministic fake LLM client returning Anthropic-shaped responses.'''
    def __init__(self, model='claude-opus-4-7'):
        self.model = model
    def respond(self, prompt: str):
        ip = max(1, len(prompt) // 4)
        op = 24
        return types.SimpleNamespace(
            model=self.model,
            usage=types.SimpleNamespace(input_tokens=ip, output_tokens=op),
            content=[types.SimpleNamespace(text=f'(mock answer) {prompt[:60]}')],
        )

def run_toy_agent(c: Chronicle, query: str = 'vegan lasagna') -> str:
    prompt = c.record('prompt', actor='user', payload={'text': f'find me a {query} recipe'})

    client = MockLLM('claude-opus-4-7')

    @tool('recipe_search')
    def recipe_search(q: str) -> list:
        corpus = {
            'vegan lasagna': ['tofu ricotta', 'cashew bechamel', 'spinach', 'zucchini'],
            'chicken curry': ['onions', 'garlic', 'garam masala'],
        }
        for k, v in corpus.items():
            if all(w in k for w in q.lower().split()):
                return v
        return []

    @llm(model='claude-opus-4-7')
    def ask(text: str):
        return client.respond(text)

    with span(c, actor='agent:planner', parent=prompt.id) as s:
        s.thought(f'the user wants a {query} recipe; I will search the corpus')
        hits = recipe_search(query)
        s.thought(f'found {len(hits)} ingredients; asking the LLM to compose')
        ask(f'Compose a short {query} recipe using: ' + ', '.join(hits))
        answer_id = s.answer(f"Here's a {query} using: {', '.join(hits)}")

    return answer_id

c1 = Chronicle()
ans1 = run_toy_agent(c1)
print(f'recorded {len(c1)} events; root = {c1.root()}')
print()
for evt in c1:
    parents = ','.join(p[:6] for p in evt.parents) or '-'
    text = str(evt.payload.get('text') or evt.payload.get('name') or '')[:60]
    dur = evt.telemetry.get('duration_ms')
    dur_s = f' ({dur:.1f}ms)' if dur else ''
    print(f'  {evt.id[:6]}  <- [{parents:<13}]  {evt.kind:<12} {evt.actor:<22} {text}{dur_s}')


### Check 2 — cross-run identity

In [ ]:
c2 = Chronicle()
ans2 = run_toy_agent(c2)
assert c1.root() == c2.root(), 'two independent runs should produce the same Merkle root'
assert set(c1.events) == set(c2.events), 'event id sets must match'
print(f'PASS — identical Merkle root across runs: {c1.root()}')


### Check 3 — tamper detection

In [ ]:
from chronicle import Event
import copy
c_tamper = Chronicle()
run_toy_agent(c_tamper)

# Swap the value at one key for a different Event (simulating an attacker
# rewriting a thought). The key (= original id) no longer matches the value.
victim_id = next(iter(c_tamper._events))
forged = Event(
    kind='thought', actor='attacker',
    payload={'text': 'malicious replacement'},
)
c_tamper._events[victim_id] = forged

ok, bad = c_tamper.verify()
assert not ok, 'verify() should reject tampered store'
assert victim_id in bad
print(f'PASS — tamper detected at {victim_id[:12]} (forged.id={forged.id[:12]})')


### Check 4 — causal query (`why()`)

In [ ]:
# Re-run a clean Chronicle for the causal query
c = Chronicle()
ans_id = run_toy_agent(c)

trace = c.why(ans_id)
kinds = [e.kind for e in trace]
actors = [e.actor for e in trace]
print('causal lineage of the answer:')
for e in trace:
    print(f'  {e.kind:<12} {e.actor:<18} {str(e.payload)[:60]}')

# The answer was caused by: prompt -> thought -> tool_call -> tool_result
#                          -> thought -> llm_call -> llm_result -> answer
expected = ['prompt', 'thought', 'tool_call', 'tool_result',
            'thought', 'llm_call', 'llm_result', 'answer']
assert kinds == expected, f'expected {expected}, got {kinds}'
print('\nPASS — causal path matches the expected reasoning shape')


### Check 5 — semantic recall

In [ ]:
hits = c.search('vegan recipe ingredient assembly', k=3)
print('top semantic matches:')
for evt, score in hits:
    print(f'  {score:+.3f}  {evt.kind:<12} {evt.actor:<18} {str(evt.payload)[:60]}')

# At least one of the top-3 should be a planner thought or answer about lasagna
top_kinds = {evt.kind for evt, _ in hits}
assert top_kinds & {'thought', 'answer', 'tool_result'}, top_kinds
print('\nPASS — semantic search surfaces meaning-relevant events')


### Check 6 — counterfactual branching

In [ ]:
# Branch at the first planner thought, then run a different continuation.
first_thought = next(e for e in c if e.kind == 'thought')
branch = c.branch_at(first_thought.id)

with span(branch, actor='agent:planner', parent=first_thought.id) as s:
    s.thought('alternative: ask the user to clarify dietary constraints first')
    s.answer('Before I search: any nut allergies or gluten preferences?')

d = Chronicle.diff(c, branch)
print(f'common events:      {len(d["common"])}')
print(f'only in original:   {len(d["only_in_a"])}')
print(f'only in branch:     {len(d["only_in_b"])}')

# Common events must include the prompt + first thought (the shared prefix)
assert first_thought.id in d['common']
# The branch must contain at least the two new events
assert len(d['only_in_b']) >= 2
# Diff is exact set algebra — no overlap
assert set(d['only_in_a']).isdisjoint(d['only_in_b'])
print('\nPASS — counterfactual diff is exact')


### DAG visualization

In [ ]:
# Render the original DAG and the branched DAG side-by-side.
import matplotlib.pyplot as plt
import networkx as nx

def to_nx(ch: Chronicle) -> nx.DiGraph:
    g = nx.DiGraph()
    for evt in ch:
        g.add_node(evt.id[:6], label=f'{evt.kind}\n{evt.actor}')
        for p in evt.parents:
            g.add_edge(p[:6], evt.id[:6])
    return g

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ch, title in zip(axes, [c, branch], ['original', 'counterfactual branch']):
    g = to_nx(ch)
    try:
        pos = nx.nx_agraph.graphviz_layout(g, prog='dot')
    except Exception:
        pos = nx.spring_layout(g, seed=42)
    nx.draw_networkx_nodes(g, pos, ax=ax, node_size=1600, node_color='#cfe2ff', edgecolors='#1f4e79')
    nx.draw_networkx_edges(g, pos, ax=ax, arrowsize=15, edge_color='#555')
    labels = {n: g.nodes[n]['label'] for n in g.nodes}
    nx.draw_networkx_labels(g, pos, labels=labels, ax=ax, font_size=7)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


### Check 7 — JSONL round-trip

In [ ]:
import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.jsonl', delete=False) as f:
    path = f.name
c.dump(path)
loaded = Chronicle.load(path)
assert loaded.root() == c.root(), 'roundtripped Chronicle must have the same root'
ok, bad = loaded.verify()
assert ok, bad
os.unlink(path)
print(f'PASS — JSONL round-trip preserves the Merkle root: {c.root()}')


### Check 8 — telemetry rollup (`stats()`)

In [ ]:
from pprint import pprint
stats = c.stats()
print(f'events              {stats["total_events"]}')
print(f'llm_calls           {stats["llm_calls"]}')
print(f'tool_calls          {stats["tool_calls"]}')
print(f'input_tokens        {stats["input_tokens"]}')
print(f'output_tokens       {stats["output_tokens"]}')
print(f'llm_duration_ms     {stats["llm_duration_ms"]:.2f}')
print(f'tool_duration_ms    {stats["tool_duration_ms"]:.2f}')
print(f'estimated_cost_usd  ${stats["estimated_cost_usd"]:.6f}')
print()
print('by_kind:')
for k, v in stats['by_kind'].items():
    print(f'  {k:<14} {v}')
print()
print('by_model:')
for m, row in stats['by_model'].items():
    print(f'  {m}')
    for k, v in row.items():
        print(f'    {k:<16} {v}')

assert stats['llm_calls'] >= 1, 'expected at least one llm_call'
assert stats['tool_calls'] >= 1
assert stats['input_tokens'] > 0
assert stats['estimated_cost_usd'] >= 0.0
print()
print('PASS - telemetry rollup populated')


## §12 Dashboard preview (in-notebook render)

Streamlit can't run inside a notebook cell (it serves a web app), so here we render the **same data the dashboard shows**, using matplotlib + a pandas table. After the wheel is installed, run `chronicle-dashboard trace.jsonl` to get the interactive web UI.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# LLM-calls table (sorted by duration desc)
llm_rows = []
for e in c:
    if e.kind == 'llm_result':
        llm_rows.append({
            'id': e.id[:8],
            'model': e.payload.get('model'),
            'input_tokens': e.payload.get('input_tokens', 0),
            'output_tokens': e.payload.get('output_tokens', 0),
            'duration_ms': round(e.telemetry.get('duration_ms', 0.0), 3),
            'cost_usd': round(e.telemetry.get('cost_usd', 0.0), 6),
            'text': (e.payload.get('text') or '')[:60],
        })
if llm_rows:
    print('LLM CALLS')
    print(pd.DataFrame(llm_rows).to_string(index=False))
    print()

# Tool-calls table
tool_rows = []
for e in c:
    if e.kind == 'tool_result':
        tool_rows.append({
            'id': e.id[:8],
            'tool': e.payload.get('name'),
            'duration_ms': round(e.telemetry.get('duration_ms', 0.0), 3),
            'result': str(e.payload.get('result'))[:60],
        })
if tool_rows:
    print('TOOL CALLS')
    print(pd.DataFrame(tool_rows).to_string(index=False))


In [ ]:
# Event-kind histogram + per-model token bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

by_kind = c.stats()['by_kind']
axes[0].bar(list(by_kind.keys()), list(by_kind.values()), color='#4c78a8')
axes[0].set_title('Event kinds')
axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=35)

by_model = c.stats()['by_model']
if by_model:
    models = list(by_model.keys())
    inp = [by_model[m]['input_tokens'] for m in models]
    out = [by_model[m]['output_tokens'] for m in models]
    x = range(len(models))
    axes[1].bar(x, inp, label='input', color='#54a24b')
    axes[1].bar(x, out, bottom=inp, label='output', color='#e45756')
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(models, rotation=20)
    axes[1].set_title('Tokens by model')
    axes[1].set_ylabel('tokens')
    axes[1].legend()

plt.tight_layout()
plt.show()


### Launching the interactive dashboard

After the wheel is installed (next section), dump any Chronicle to JSONL and launch:

```bash
chronicle-dashboard /path/to/trace.jsonl
```

or from Python:

```python
from chronicle.dashboard import launch
launch("/path/to/trace.jsonl")
```

It opens at <http://localhost:8501>.


## §13 Build the wheel

Now that all checks pass on the in-tree source, we write `pyproject.toml`, ensure `README.md` exists (the human-curated one in the repo is the source of truth — we only write a fallback stub if it's missing), invoke `python -m build --wheel`, and confirm the artifact lands in `dist/`.


In [ ]:
_SRC = r'''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "chronicle"
version = "0.2.0"
description = "Merkle-DAG agent traceability with causal, integrity, semantic, and telemetry views."
readme = "README.md"
requires-python = ">=3.10"
license = { text = "MIT" }
authors = [{ name = "Sanjay Babu" }]
dependencies = [
    "numpy>=1.24",
]

[project.optional-dependencies]
st = ["sentence-transformers>=2.2"]
dashboard = [
    "streamlit>=1.30",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "networkx>=3.0",
]

[project.scripts]
chronicle-dashboard = "chronicle.dashboard:cli"

[tool.hatch.build.targets.wheel]
packages = ["src/chronicle"]
'''
from pathlib import Path
_p = Path('pyproject.toml'); _p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(_SRC, encoding='utf-8')
print(f'wrote {_p} ({len(_SRC)} bytes)')


In [ ]:
# Don't overwrite the hand-curated README at the repo root. Only write a
# minimal stub when it's genuinely missing (e.g. on a fresh checkout).
from pathlib import Path
_readme = Path('README.md')
if not _readme.exists():
    _readme.write_text('# chronicle\n\nMerkle-DAG agent traceability. Built from `notebook/chronicle.ipynb`.\n\n```python\nimport chronicle\nc = chronicle.demo()\nprint("root:", c.root())\nprint("verify:", c.verify())\nfor evt, score in c.search("vegan recipe"):\n    print(f"{score:+.3f}  {evt.kind:<12} {evt.actor}")\n```\n\nSee the notebook for the design and the headline properties:\ndeterministic cross-run identity, tamper detection, causal `why()`,\nsemantic search, and counterfactual branching.\n', encoding='utf-8')
    print(f'wrote stub {_readme}')
else:
    print(f'kept existing {_readme} ({_readme.stat().st_size:,} bytes)')


In [ ]:
import subprocess, sys
# Use the *current* interpreter so the wheel is built in the active venv.
r = subprocess.run(
    [sys.executable, '-m', 'build', '--wheel', '--no-isolation'],
    capture_output=True, text=True
)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise SystemExit(r.returncode)

from pathlib import Path
wheels = sorted(Path('dist').glob('chronicle-*.whl'))
assert wheels, 'no wheel was produced'
WHEEL = wheels[-1]
print('built:', WHEEL)


## §14 Install the wheel and import it

We install the just-built wheel into the active Python and reload `chronicle` from site-packages (rather than the src tree we've been using all along). If `demo()` runs, `verify()` passes, and `stats()` reports the expected events, the wheel is sound.


In [ ]:
import subprocess, sys, importlib
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', '--quiet', str(WHEEL)],
    capture_output=True, text=True,
)
print(r.stdout, r.stderr)
assert r.returncode == 0

# Drop the src/ shim from sys.path so we import the *installed* package.
sys.path = [p for p in sys.path if 'src' not in p.split('/')[-2:]]
for mod in list(sys.modules):
    if mod == 'chronicle' or mod.startswith('chronicle.'):
        del sys.modules[mod]
import chronicle
print('imported chronicle', chronicle.__version__, 'from', chronicle.__file__)

c = chronicle.demo()
ok, bad = c.verify()
assert ok, bad
s = c.stats()
print(f'demo Chronicle: {len(c)} events, root={c.root()}, verify=OK')
print(f'  llm_calls={s["llm_calls"]}  tool_calls={s["tool_calls"]}  tokens={s["total_tokens"]}')
print(f'  llm_latency={s["llm_duration_ms"]:.1f}ms  est_cost=${s["estimated_cost_usd"]:.6f}')
for evt, score in c.search('vegan recipe ingredients'):
    print(f'  {score:+.3f}  {evt.kind:<12} {evt.actor:<22} {str(evt.payload)[:60]}')


### Dump a JSONL trace for the dashboard to load

After this cell runs, `demo_trace.jsonl` exists at the project root. From a shell:

```bash
chronicle-dashboard demo_trace.jsonl
```


In [ ]:
from pathlib import Path
trace_path = Path('demo_trace.jsonl').resolve()
c.dump(trace_path)
print(f'wrote {trace_path} ({trace_path.stat().st_size} bytes)')
print('to launch the dashboard:')
print(f'  chronicle-dashboard {trace_path}')


## §15 Research notes & next steps

**The contribution.** Treating an agent trace as a content-addressed causal DAG with a semantic overlay collapses *causality, integrity, cross-run comparison, search, and counterfactual replay* into a single substrate, rather than requiring orthogonal subsystems for each. The headline property is **deterministic cross-run identity** — identical reasoning produces identical ids, making A/B comparison of agent variants exact.

**Limitations of this prototype.**
- Only the hashing-trick embedder is exercised in the smoke test; the   `SentenceTransformerEmbedder` adapter is included but optional.
- In-memory only. Persistence is JSONL — fine for research scale, not for   long-running production agents.
- No cryptographic signatures yet. Tamper detection is integrity-only;   add signed Merkle roots (Ed25519) for adversarial-robustness work.
- No OpenTelemetry bridge. Adding an exporter would let traces flow into   Jaeger / Tempo / Langfuse for cross-comparison with span-tree tools.

**Citing this design.** If you publish, the framing to lean on is: "agent traces as Merkle DAGs with semantic overlay — a unified substrate for causality, integrity, and counterfactual analysis". As far as I can find, this combined framing is not present in the existing agent-observability literature.
